# 07 Dense Retrieval

Σε αυτό το notebook εκτελείται το dense retrieval baseline. Οι ερωτήσεις αντιστοιχίζονται στα διαθέσιμα chunks μέσω των embeddings και υπολογίζονται τα βασικά retrieval αποτελέσματα.


In [ ]:
# Uncomment if needed
# !pip install -q sentence-transformers faiss-cpu pyarrow tqdm

In [ ]:
from pathlib import Path
import json
import warnings
import re

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import faiss
import torch
from sentence_transformers import SentenceTransformer


In [ ]:
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 180)

IN_KAGGLE = Path("/kaggle/working").exists()
print("IN_KAGGLE:", IN_KAGGLE)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
EMBEDDING_MODEL = "BAAI/bge-m3"
TOP_K = 10

USE_QUERY_LIMIT = False
QUERY_LIMIT = 50

retrieval_config = {
    "retrieval_type": "dense",
    "embedding_model": EMBEDDING_MODEL,
    "top_k": TOP_K,
    "document_known": False,
    "query_expansion": True
}

retrieval_config

In [ ]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

DATA_DIR = BASE_DIR / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

CHUNKS_DIR = PROCESSED_DIR / "chunks"
EMBEDDINGS_DIR = PROCESSED_DIR / "embeddings"
RETRIEVAL_DIR = PROCESSED_DIR / "retrieval_results"

WORKING_DATASET_CSV_PATH = INTERIM_DIR / "financebench_open_source_working.csv"
WORKING_DATASET_PARQUET_PATH = INTERIM_DIR / "financebench_open_source_working.parquet"

CHUNKS_CSV_PATH = CHUNKS_DIR / "financebench_chunks.csv"
CHUNKS_PARQUET_PATH = CHUNKS_DIR / "financebench_chunks.parquet"

EMBEDDINGS_MATRIX_PATH = EMBEDDINGS_DIR / "chunk_embeddings.npy"
EMBEDDINGS_METADATA_CSV_PATH = EMBEDDINGS_DIR / "chunk_embeddings_metadata.csv"
EMBEDDINGS_METADATA_PARQUET_PATH = EMBEDDINGS_DIR / "chunk_embeddings_metadata.parquet"

RETRIEVAL_RESULTS_CSV_PATH = RETRIEVAL_DIR / "retrieval_results_dense.csv"
RETRIEVAL_RESULTS_PARQUET_PATH = RETRIEVAL_DIR / "retrieval_results_dense.parquet"
RETRIEVAL_MANIFEST_PATH = RETRIEVAL_DIR / "retrieval_manifest_dense.csv"
RETRIEVAL_STATS_PATH = RETRIEVAL_DIR / "retrieval_stats_dense.json"

RETRIEVAL_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("RETRIEVAL_DIR:", RETRIEVAL_DIR)

In [ ]:
if WORKING_DATASET_PARQUET_PATH.exists():
    query_df = pd.read_parquet(WORKING_DATASET_PARQUET_PATH)
elif WORKING_DATASET_CSV_PATH.exists():
    query_df = pd.read_csv(WORKING_DATASET_CSV_PATH)
else:
    raise FileNotFoundError("Working dataset not found.")

print("query_df shape before filtering:", query_df.shape)
query_df.head(2)

In [ ]:
required_cols = ["question", "financebench_id"]

missing_cols = [c for c in required_cols if c not in query_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in query_df: {missing_cols}")

query_df = query_df.copy()

# create question_clean if missing
if "question_clean" not in query_df.columns:
    query_df["question_clean"] = query_df["question"].astype(str)

query_df["question_clean"] = (
    query_df["question_clean"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

query_df = query_df[query_df["question_clean"].notna()].copy().reset_index(drop=True)

if USE_QUERY_LIMIT:
    query_df = query_df.head(QUERY_LIMIT).copy().reset_index(drop=True)

print("query_df shape after filtering:", query_df.shape)
print(query_df.columns.tolist())
query_df[["financebench_id", "question", "question_clean"]].head(3)

In [ ]:
if CHUNKS_PARQUET_PATH.exists():
    chunks_df = pd.read_parquet(CHUNKS_PARQUET_PATH)
elif CHUNKS_CSV_PATH.exists():
    chunks_df = pd.read_csv(CHUNKS_CSV_PATH)
else:
    raise FileNotFoundError("Chunks file not found.")

print("chunks_df shape:", chunks_df.shape)
print(chunks_df.columns.tolist())
chunks_df.head(2)

In [ ]:

print("BASE_DIR:", BASE_DIR)
print("CHUNKS_DIR exists:", CHUNKS_DIR.exists())
print("CHUNKS_DIR:", CHUNKS_DIR)

if CHUNKS_DIR.exists():
    print("Files inside CHUNKS_DIR:")
    for p in sorted(CHUNKS_DIR.iterdir()):
        print("-", p.name)


In [ ]:
if EMBEDDINGS_METADATA_PARQUET_PATH.exists():
    embeddings_metadata_df = pd.read_parquet(EMBEDDINGS_METADATA_PARQUET_PATH)
elif EMBEDDINGS_METADATA_CSV_PATH.exists():
    embeddings_metadata_df = pd.read_csv(EMBEDDINGS_METADATA_CSV_PATH)
else:
    raise FileNotFoundError("Embeddings metadata file not found.")

if not EMBEDDINGS_MATRIX_PATH.exists():
    raise FileNotFoundError("Embeddings matrix .npy file not found.")

embeddings_matrix = np.load(EMBEDDINGS_MATRIX_PATH)

print("embeddings_metadata_df shape:", embeddings_metadata_df.shape)
print("embeddings_matrix shape:", embeddings_matrix.shape)

assert len(embeddings_metadata_df) == len(embeddings_matrix), "Metadata and embeddings row count mismatch"

print(embeddings_metadata_df.columns.tolist())
embeddings_metadata_df.head(2)

In [ ]:
retrieval_corpus_df = chunks_df.copy().reset_index(drop=True)

assert len(retrieval_corpus_df) == len(embeddings_matrix), \
    "Chunks dataframe and embeddings matrix row count mismatch"

print("retrieval_corpus_df shape:", retrieval_corpus_df.shape)
print(retrieval_corpus_df.columns.tolist())
retrieval_corpus_df.head(2)

In [ ]:
required_chunk_cols = ["chunk_id", "doc_id", "chunk_text"]

missing_chunk_cols = [c for c in required_chunk_cols if c not in retrieval_corpus_df.columns]
if missing_chunk_cols:
    raise ValueError(f"Missing required columns in retrieval_corpus_df: {missing_chunk_cols}")

print("Retrieval corpus columns OK.")

In [ ]:
FINANCE_ALIAS_MAP = {
    "ppe": [
        "property plant equipment",
        "property plant and equipment",
        "property plant and equipment net",
        "pp&e",
        "balance sheet"
    ],
    "net ppe": [
        "property plant and equipment net",
        "net property plant equipment",
        "property plant equipment net",
        "balance sheet"
    ],
    "capex": [
        "capital expenditure",
        "capital expenditures",
        "purchases of property plant and equipment",
        "purchases of property, plant and equipment",
        "cash flow statement"
    ],
    "cogs": [
        "cost of goods sold",
        "cost of sales",
        "cost of products sold",
        "income statement"
    ],
    "ebitda": [
        "earnings before interest taxes depreciation and amortization",
        "non-gaap operating performance"
    ],
    "opex": [
        "operating expenses",
        "selling general and administrative",
        "sg&a"
    ],
    "gross margin": [
        "gross profit margin",
        "gross profit",
        "income statement"
    ],
    "operating cash flow": [
        "net cash provided by operating activities",
        "cash flow statement"
    ],
    "free cash flow": [
        "free cash flow",
        "net cash provided by operating activities",
        "capital expenditures"
    ],
    "working capital": [
        "current assets",
        "current liabilities",
        "balance sheet"
    ],
    "payout ratio": [
        "dividends",
        "net income",
        "dividend payout ratio"
    ],
    "retention ratio": [
        "retained earnings ratio",
        "dividend payout ratio",
        "dividends",
        "net income"
    ],
}

In [ ]:
def normalize_query_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def expand_finance_query(query: str) -> str:
    q = normalize_query_text(query)
    expansions = []

    ordered_aliases = sorted(FINANCE_ALIAS_MAP.keys(), key=len, reverse=True)

    for alias in ordered_aliases:
        if alias in q:
            expansions.extend(FINANCE_ALIAS_MAP[alias])

    if "balance sheet" in q:
        expansions.extend(["balance sheet", "assets liabilities equity"])
    if "cash flow" in q:
        expansions.extend(["cash flow statement", "operating activities investing activities financing activities"])
    if "income statement" in q:
        expansions.extend(["income statement", "net sales operating income net income"])
    if "year end" in q or "year-end" in q:
        expansions.extend(["at december 31", "balance sheet"])

    seen = set()
    cleaned_expansions = []

    for item in expansions:
        item = item.strip().lower()
        if item and item not in seen:
            seen.add(item)
            cleaned_expansions.append(item)

    if cleaned_expansions:
        return q + " " + " ".join(cleaned_expansions)

    return q

In [ ]:
query_df["question_clean"] = query_df["question_clean"].astype(str)
query_df["expanded_question"] = query_df["question_clean"].apply(expand_finance_query)

query_df[["financebench_id", "question_clean", "expanded_question"]].head(10)

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print("Loaded embedding model:", EMBEDDING_MODEL)

In [ ]:
def embed_queries(texts):
    vectors = embedding_model.encode(
        texts,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    return np.asarray(vectors, dtype="float32")

In [ ]:
def build_global_index(embeddings: np.ndarray):
    vectors = embeddings.astype("float32").copy()
    faiss.normalize_L2(vectors)

    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index


def search_global(query_vector: np.ndarray, index, top_k: int):
    scores, indices = index.search(query_vector, k=min(top_k, index.ntotal))
    return scores[0], indices[0]

In [ ]:
global_index = build_global_index(embeddings_matrix)
print("Global FAISS index size:", global_index.ntotal)

In [ ]:
query_vectors = embed_queries(query_df["expanded_question"].tolist())
print("query_vectors shape:", query_vectors.shape)

In [ ]:
retrieval_records = []
manifest_records = []

for query_idx, (_, row) in enumerate(
    tqdm(query_df.iterrows(), total=len(query_df), desc="Running dense retrieval (document-unknown)")
):
    financebench_id = row.get("financebench_id")
    expected_doc_name = row.get("doc_name")
    question = row["question"]
    question_clean = row["question_clean"]
    expanded_question = row["expanded_question"]

    manifest_record = {
        "query_row": query_idx,
        "financebench_id": financebench_id,
        "question": question,
        "expected_doc_name": expected_doc_name,
        "status": None,
        "error_message": None,
        "n_results": 0
    }

    try:
        qvec = query_vectors[query_idx].reshape(1, -1)
        scores, global_indices = search_global(
            query_vector=qvec,
            index=global_index,
            top_k=TOP_K
        )

        for rank, (score, global_idx) in enumerate(zip(scores, global_indices), start=1):
            matched_row = retrieval_corpus_df.iloc[int(global_idx)]

            retrieval_records.append({
            "query_row": query_idx,
             "financebench_id": financebench_id,
              "question": question,
             "question_clean": question_clean,
              "expanded_question": expanded_question,
              "expected_doc_name": expected_doc_name,
              "expected_company": row.get("company"),
              "retrieved_rank": rank,
             "retrieval_score": float(score),
               "embedding_row_idx": int(global_idx),
              "chunk_id": matched_row["chunk_id"],
              "retrieved_doc_id": matched_row["doc_id"],
              "chunk_index": matched_row.get("chunk_index"),
               "chunk_text": matched_row["chunk_text"],
              "char_count": matched_row.get("char_count"),
               "token_estimate": matched_row.get("token_estimate"),
                })

        manifest_record["status"] = "success"
        manifest_record["n_results"] = len(global_indices)

    except Exception as e:
        manifest_record["status"] = "error"
        manifest_record["error_message"] = str(e)

    manifest_records.append(manifest_record)

retrieval_results_df = pd.DataFrame(retrieval_records)
retrieval_manifest_df = pd.DataFrame(manifest_records)

print("retrieval_results_df shape:", retrieval_results_df.shape)
print("retrieval_manifest_df shape:", retrieval_manifest_df.shape)

In [ ]:
retrieval_results_df["doc_match"] = (
    retrieval_results_df["expected_doc_name"].fillna("").astype(str)
    == retrieval_results_df["retrieved_doc_id"].fillna("").astype(str)
)

retrieval_results_df[[
    "financebench_id",
    "retrieved_rank",
    "retrieved_doc_id",
    "expected_doc_name",
    "doc_match",
    "retrieval_score"
]].head(15)

In [ ]:
top1_df = retrieval_results_df[retrieval_results_df["retrieved_rank"] == 1].copy()

top1_doc_match_rate = float(top1_df["doc_match"].mean()) if len(top1_df) else 0.0
topk_doc_match_rate = float(
    retrieval_results_df.groupby("financebench_id")["doc_match"].max().mean()
) if len(retrieval_results_df) else 0.0

summary_df = pd.DataFrame([{
    "n_queries": int(query_df["financebench_id"].nunique()),
    "top1_doc_match_rate": top1_doc_match_rate,
    f"top{TOP_K}_doc_match_rate": topk_doc_match_rate
}])

summary_df

In [ ]:
retrieval_results_df.to_csv(RETRIEVAL_RESULTS_CSV_PATH, index=False, encoding="utf-8")
retrieval_results_df.to_parquet(RETRIEVAL_RESULTS_PARQUET_PATH, index=False)

retrieval_manifest_df.to_csv(RETRIEVAL_MANIFEST_PATH, index=False, encoding="utf-8")
print("Αποθηκεύτηκαν:")
print("-", RETRIEVAL_RESULTS_CSV_PATH)
print("-", RETRIEVAL_RESULTS_PARQUET_PATH)
print("-", RETRIEVAL_MANIFEST_PATH)


In [ ]:
retrieval_stats = {
    "retrieval_type": "dense",
    "document_known": False,
    "query_expansion": True,
    "embedding_model": EMBEDDING_MODEL,
    "top_k": TOP_K,
    "n_queries": int(query_df["financebench_id"].nunique()),
    "n_result_rows": int(len(retrieval_results_df)),
    "n_manifest_rows": int(len(retrieval_manifest_df)),
    "top1_doc_match_rate": top1_doc_match_rate,
    f"top{TOP_K}_doc_match_rate": topk_doc_match_rate,
    "results_csv": str(RETRIEVAL_RESULTS_CSV_PATH),
    "results_parquet": str(RETRIEVAL_RESULTS_PARQUET_PATH),
    "manifest_csv": str(RETRIEVAL_MANIFEST_PATH),
}

with open(RETRIEVAL_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(retrieval_stats, f, indent=2, ensure_ascii=False)

print("Saved stats:", RETRIEVAL_STATS_PATH)
retrieval_stats

In [ ]:
retrieval_results_df[[
    "financebench_id",
    "question",
    "expanded_question",
    "retrieved_rank",
    "retrieved_doc_id",
    "expected_doc_name",
    "doc_match",
    "retrieval_score",
    "chunk_id"
]].head(20)

## Συμπέρασμα

Σε αυτό το notebook:

- εκτελέστηκε dense retrieval baseline
- χρησιμοποιήθηκε document-known retrieval
- η αναζήτηση περιορίστηκε στο σωστό report για κάθε query
- αποθηκεύτηκαν retrieval results και manifest

Το επόμενο notebook θα είναι το hybrid retrieval baseline.